# Hyperliquid BTC Hourly OHLC Analysis

Fetch hourly perp OHLC data from Hyperliquid.

In [ ]:
# Import required libraries
from clients.hyperliquid_client import HyperliquidClient
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np

In [ ]:
# Configuration
EXCHANGE = 'hyperliquid'
INTERVAL = '1m'
LOOKBACK_DAYS = 3 # NOTE: Hyperliquid's candleSnapshot API retains only ~3-4 days of 1m history.

# Static symbol universe (explicitly user-defined)
SYMBOL_UNIVERSE = ['ETH', 'SOL', 'BTC', 'XRP', 'xyz:GOLD', 'xyz:SILVER']
end_dt   = datetime.now(timezone.utc)
start_dt = end_dt - timedelta(days=LOOKBACK_DAYS)

start_date = start_dt.strftime("%Y-%m-%d")
end_date   = end_dt.strftime("%Y-%m-%d")
client = HyperliquidClient()


print(f"Fetching static universe ({INTERVAL}) from {start_date} to {end_date}:")
print(", ".join(SYMBOL_UNIVERSE))


In [ ]:
market_ohlc = {}
failed_symbols = []

for symbol in SYMBOL_UNIVERSE:
    df_symbol = client.download_ohlc(
        symbol=symbol,
        interval=INTERVAL,
        start_date=start_date,
        end_date=end_date,
    )
    if df_symbol is not None and not df_symbol.empty:
        market_ohlc[symbol] = df_symbol.copy()
    else:
        failed_symbols.append(symbol)
        print(f"  {symbol}: no data")

if failed_symbols:
    print("\nSymbols skipped (no data returned):")
    print(", ".join(failed_symbols))

if not market_ohlc:
    raise ValueError("No OHLC data was fetched for any symbols in SYMBOL_UNIVERSE.")

SYMBOL = list(market_ohlc.keys())[0]
print(f"Primary chart symbol: {SYMBOL}")

In [ ]:
market_bars = {}

for symbol, df_ohlc in market_ohlc.items():
    df_1m_temp = (
        df_ohlc.copy()
        .sort_values("timestamp")
        .reset_index(drop=True)
    )
    df_1m_temp["timestamp"] = pd.to_datetime(df_1m_temp["timestamp"], utc=True)

    df_15m_temp = (
        df_1m_temp.resample("15min", on="timestamp")
        .agg(
            open=("open", "first"),
            high=("high", "max"),
            low=("low", "min"),
            close=("close", "last"),
            volume=("volume", "sum"),
        )
        .dropna()
        .reset_index()
    )

    market_bars[symbol] = {
        "df_1m": df_1m_temp,
        "df_15m": df_15m_temp,
    }

if SYMBOL in market_bars:
    df_1m = market_bars[SYMBOL]["df_1m"]
    df_15m = market_bars[SYMBOL]["df_15m"]

## Strategy Parameters & Tuning

In [ ]:
def open_window_flag(ts_utc, tz_name, open_h, open_m, window_min):
    """Major global cash-market opens (DST-safe via timezone conversion)"""
    local_ts = ts_utc.dt.tz_convert(ZoneInfo(tz_name))
    mins = local_ts.dt.hour * 60 + local_ts.dt.minute
    start = open_h * 60 + open_m
    end = start + window_min
    return (mins >= start) & (mins < end)

def nearest_key_level_live(levels_df, now_ts, entry_price, direction):
    """Select nearest currently-open key level for TP (Take), using only info known by now_ts."""
    if levels_df.empty:
        return np.nan

    open_now = levels_df[
        (levels_df["created_at"] <= now_ts)
        & (levels_df["swept_at"].isna() | (levels_df["swept_at"] > now_ts))
    ]
    if open_now.empty:
        return np.nan

    if direction == "LONG":
        candidates = open_now[(open_now["side"] == "high") & (open_now["price"] > entry_price)]
        return candidates["price"].min() if not candidates.empty else np.nan

    candidates = open_now[(open_now["side"] == "low") & (open_now["price"] < entry_price)]
    return candidates["price"].max() if not candidates.empty else np.nan

In [ ]:
def _levels_from_row(row: pd.Series):
    return [
        # ICT session pools
        ("prev_asia_high",   row.get("prev_asia_high"),   "high", row.get("prev_asia_anchor")),
        ("prev_asia_low",    row.get("prev_asia_low"),    "low",  row.get("prev_asia_anchor")),
        ("prev_london_high", row.get("prev_london_high"), "high", row.get("prev_london_anchor")),
        ("prev_london_low",  row.get("prev_london_low"),  "low",  row.get("prev_london_anchor")),
        ("prev_ny_high",     row.get("prev_ny_high"),     "high", row.get("prev_ny_anchor")),
        ("prev_ny_low",      row.get("prev_ny_low"),      "low",  row.get("prev_ny_anchor")),

        # Rolling block pools
        ("prev_2h_high",     row.get("prev_2h_high"),     "high", row.get("prev_2h_anchor")),
        ("prev_2h_low",      row.get("prev_2h_low"),      "low",  row.get("prev_2h_anchor")),
        ("prev_4h_high",     row.get("prev_4h_high"),     "high", row.get("prev_4h_anchor")),
        ("prev_4h_low",      row.get("prev_4h_low"),      "low",  row.get("prev_4h_anchor")),
        ("prev_8h_high",     row.get("prev_8h_high"),     "high", row.get("prev_8h_anchor")),
        ("prev_8h_low",      row.get("prev_8h_low"),      "low",  row.get("prev_8h_anchor")),
        ("prev_12h_high",    row.get("prev_12h_high"),    "high", row.get("prev_12h_anchor")),
        ("prev_12h_low",     row.get("prev_12h_low"),     "low",  row.get("prev_12h_anchor")),
        ("prev_24h_high",    row.get("prev_24h_high"),    "high", row.get("prev_24h_anchor")),
        ("prev_24h_low",     row.get("prev_24h_low"),     "low",  row.get("prev_24h_anchor")),
        ("prev_36h_high",    row.get("prev_36h_high"),    "high", row.get("prev_36h_anchor")),
        ("prev_36h_low",     row.get("prev_36h_low"),     "low",  row.get("prev_36h_anchor")),
        ("prev_48h_high",    row.get("prev_48h_high"),    "high", row.get("prev_48h_anchor")),
        ("prev_48h_low",     row.get("prev_48h_low"),     "low",  row.get("prev_48h_anchor")),

        # Previous day + midnight open
        ("pdh", row.get("pdh"), "high", row.get("pd_anchor")),
        ("pdl", row.get("pdl"), "low",  row.get("pd_anchor")),
        ("midnight_open", row.get("midnight_open"), "high", row.get("day_start")),
        ("midnight_open", row.get("midnight_open"), "low",  row.get("day_start")),
    ]


def add_ict_liquidity_pools(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().sort_values("timestamp").reset_index(drop=True)
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True)
    out["hour"] = out["timestamp"].dt.hour
    out["date"] = out["timestamp"].dt.date

    # 1) Midnight open
    out["day_start"] = pd.to_datetime(out["date"]).dt.tz_localize("UTC")
    midnight_open = out[out["hour"] == 0].groupby("date")["open"].first().rename("midnight_open")
    out = out.merge(midnight_open, left_on="date", right_index=True, how="left")

    # 2) Session pools
    pools = {"asia": (0, 6), "london": (7, 10), "ny": (12, 15)}
    for name, (start, end) in pools.items():
        mask = (out["hour"] >= start) & (out["hour"] < end)
        stats = out[mask].groupby("date").agg(
            high_val=("high", "max"),
            low_val=("low", "min"),
            start_ts=("timestamp", "min"),
        )
        stats[f"prev_{name}_high"] = stats["high_val"].shift(1)
        stats[f"prev_{name}_low"] = stats["low_val"].shift(1)
        stats[f"prev_{name}_anchor"] = stats["start_ts"].shift(1)

        out = out.merge(
            stats[[f"prev_{name}_high", f"prev_{name}_low", f"prev_{name}_anchor"]],
            left_on="date",
            right_index=True,
            how="left",
        )
        out[f"prev_{name}_anchor"] = pd.to_datetime(out[f"prev_{name}_anchor"], utc=True)

    # helper: floor timestamp to arbitrary hour multiple via epoch nanoseconds
    def _floor_to_hours(ts_series: pd.Series, hours: int) -> pd.Series:
        ns = hours * 3_600 * 1_000_000_000
        return pd.to_datetime((ts_series.astype(np.int64) // ns) * ns, utc=True)

    # 3) Rolling block pools
    for hours in [2, 4, 8, 12, 24, 36, 48]:
        col = f"_block_{hours}h"
        out[col] = _floor_to_hours(out["timestamp"], hours)
        b = out.groupby(col).agg(
            high_val=("high", "max"),
            low_val=("low", "min"),
            start_ts=("timestamp", "min"),
        )
        b[f"prev_{hours}h_high"] = b["high_val"].shift(1)
        b[f"prev_{hours}h_low"] = b["low_val"].shift(1)
        b[f"prev_{hours}h_anchor"] = b["start_ts"].shift(1)
        out = out.merge(
            b[[f"prev_{hours}h_high", f"prev_{hours}h_low", f"prev_{hours}h_anchor"]],
            left_on=col,
            right_index=True,
            how="left",
        )
        out[f"prev_{hours}h_anchor"] = pd.to_datetime(out[f"prev_{hours}h_anchor"], utc=True)
        out = out.drop(columns=[col])

    # 4) Previous day high/low
    daily = out.groupby("date").agg(
        d_high=("high", "max"),
        d_low=("low", "min"),
        d_start=("timestamp", "min"),
    )
    daily["pdh"] = daily["d_high"].shift(1)
    daily["pdl"] = daily["d_low"].shift(1)
    daily["pd_anchor"] = daily["d_start"].shift(1)

    out = out.merge(daily[["pdh", "pdl", "pd_anchor"]], left_on="date", right_index=True, how="left")
    out["pd_anchor"] = pd.to_datetime(out["pd_anchor"], utc=True)

    return out.ffill()


def build_levels_and_sweeps_causal(df_htf_ctx: pd.DataFrame):
    htf = df_htf_ctx.copy().sort_values("timestamp").reset_index(drop=True)

    # calculate the absolute true range (ATR) for the HTF context, which is used in sweep reliability scoring
    prev_close = htf["close"].shift(1)
    tr = pd.concat([
        (htf["high"] - htf["low"]).abs(),
        (htf["high"] - prev_close).abs(),
        (htf["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    htf["atr_14"] = tr.rolling(14, min_periods=14).mean()

    htf["vol_sma_20"] = htf["volume"].rolling(20, min_periods=20).mean()
    htf["vol_std_20"] = htf["volume"].rolling(20, min_periods=20).std()
    htf["vol_z_20"] = (htf["volume"] - htf["vol_sma_20"]) / htf["vol_std_20"].replace(0, np.nan)

    levels = []
    sweeps = []
    seen = set()

    for i, row in htf.iterrows():
        ts = row["timestamp"]
        ts_high = row["high"]
        ts_low = row["low"]
        ts_close = row["close"]

        for level_type, price, side, anchor in _levels_from_row(row):
            if pd.isna(price) or pd.isna(anchor):
                # Skip levels with missing price or anchor time, as they cannot be tracked or scored reliably
                continue
            
            key = (level_type, side, float(price), pd.Timestamp(anchor))
            if key in seen:
                continue
            seen.add(key)
            levels.append({
                "level_type": level_type,
                "side": side,
                "price": float(price),
                "anchor_time": pd.Timestamp(anchor),
                "created_at": ts,
                "swept": False,
                "swept_at": pd.NaT,
                "sweep_idx": np.nan,
                "touch_count": 0,
            })

        for lvl in levels:
            level_price = lvl["price"]
            level_side = lvl["side"]

            if level_side == "high":
                touched_now = ts_high >= level_price 
                # bar closes below level price (reclaim) after reaching above it, which is a bearish sweep of the level
                swept_now = (ts_high > level_price) and (ts_close < level_price) 
                wick_excess = max(0.0, float(ts_high - level_price))
                reclaim_strength = max(0.0, float(level_price - ts_close))
                direction = "SHORT"
            else:
                touched_now = ts_low <= level_price
                swept_now = (ts_low < level_price) and (ts_close > level_price)
                wick_excess = max(0.0, float(level_price - ts_low))
                reclaim_strength = max(0.0, float(ts_close - level_price))
                direction = "LONG"

            # normalize wick excess and reclaim strength by ATR for better cross-level comparability in scoring
            atr = row["atr_14"]
            atr_valid = pd.notna(atr) and float(atr) > 0
            wick_excess_atr = (wick_excess / float(atr)) if atr_valid else np.nan
            reclaim_strength_atr = (reclaim_strength / float(atr)) if atr_valid else np.nan
            
            vol_z = row["vol_z_20"]
            vol_available = pd.notna(vol_z)

            if swept_now:
                lvl["swept"] = True
                lvl["swept_at"] = ts
                lvl["sweep_idx"] = i
                pre_sweep_touches = int(lvl.get("touch_count", 0))
                
                reliability_score = compute_sweep_reliability_score(
                    wick_excess_atr=0.0 if pd.isna(wick_excess_atr) else wick_excess_atr,
                    reclaim_strength_atr=0.0 if pd.isna(reclaim_strength_atr) else reclaim_strength_atr,
                    volume_spike_z=0.0 if pd.isna(vol_z) else float(vol_z),
                    pre_sweep_touches=pre_sweep_touches,
                    vol_available=vol_available,
                )

                sweeps.append({
                    "timestamp": ts,
                    "idx": i,
                    "direction": direction,
                    "level_type": lvl["level_type"],
                    "level_side": level_side,
                    "level_price": level_price,
                    "pre_sweep_touches": pre_sweep_touches,
                    "wick_excess": float(wick_excess),
                    "wick_excess_atr": float(wick_excess_atr) if pd.notna(wick_excess_atr) else np.nan,
                    "reclaim_strength": float(reclaim_strength),
                    "reclaim_strength_atr": float(reclaim_strength_atr) if pd.notna(reclaim_strength_atr) else np.nan,
                    "volume_spike_z": float(vol_z) if pd.notna(vol_z) else np.nan,
                    "reliability_score": reliability_score,
                })
            elif touched_now:
                lvl["touch_count"] += 1

    levels_df = pd.DataFrame(levels)
    sweeps_df = pd.DataFrame(sweeps)
    if not levels_df.empty:
        levels_df["swept_at"] = pd.to_datetime(levels_df["swept_at"], utc=True)
    return levels_df, sweeps_df

def _clip01(x):
    return max(0.0, min(1.0, float(x)))

def compute_sweep_reliability_score(
    wick_excess_atr: float,
    reclaim_strength_atr: float,
    volume_spike_z: float,
    pre_sweep_touches: int,
    vol_available: bool,
):
    """Score sweep quality from 0 to 100 using causal HTF-only features."""
    
    # each feature is clipped to [0,1] to prevent outliers from dominating the score
    # then weighted and combined into a raw score which is finally scaled to [0,100]
    wick_component = _clip01(wick_excess_atr / 1.0)
    reclaim_component = _clip01(reclaim_strength_atr / 0.6)
    volume_component = _clip01((volume_spike_z + 0.5) / 2.5) if vol_available else 0.5
    touch_component = _clip01((4.0 - min(float(pre_sweep_touches), 4.0)) / 4.0)

    raw_score = (
        0.35 * wick_component
        + 0.25 * reclaim_component
        + 0.20 * volume_component
        + 0.20 * touch_component
    )
    return float(np.clip(raw_score * 100.0, 0.0, 100.0))


In [ ]:
def add_bos_choch(df, left=3, right=3, atr_len=14, min_break_atr=0.10):
    """
    BOS/CHOCH detector for LTF candles.
    - Pivots are confirmed only after `right` bars have closed.
    - Breaks require close-through + ATR displacement filter.
    """
    out = df.copy().sort_values('timestamp').reset_index(drop=True)

    # average true range (ATR) for volatility-based break filters
    prev_close = out['close'].shift(1)
    tr = pd.concat([
        (out['high'] - out['low']).abs(),
        (out['high'] - prev_close).abs(),
        (out['low'] - prev_close).abs(),
    ], axis=1).max(axis=1)
    out['atr'] = tr.rolling(atr_len, min_periods=atr_len).mean()

    # Confirmed pivots available at current bar without look-ahead
    # candidate pivot is bar i-right; confirmation occurs at i
    win = left + right + 1
    roll_max = out['high'].rolling(win, min_periods=win).max()
    roll_min = out['low'].rolling(win, min_periods=win).min()

    candidate_high = out['high'].shift(right)
    candidate_low = out['low'].shift(right)


    out['swing_high_confirmed'] = candidate_high >= roll_max
    out['swing_low_confirmed'] = candidate_low <= roll_min

    out['swing_high_price'] = np.where(out['swing_high_confirmed'], candidate_high, np.nan)
    out['swing_low_price'] = np.where(out['swing_low_confirmed'], candidate_low, np.nan)

    out['last_swing_high'] = out['swing_high_price'].ffill()
    out['last_swing_low'] = out['swing_low_price'].ffill()

    # Close-through break with displacement filter to avoid false signals from weak breaks that don't show real commitment
    # upward break detection
    up_break = (
        (out['close'] > out['last_swing_high']) # close above last swing high
        & (out['close'].shift(1) <= out['last_swing_high']) # Previous close at or below last swing high
        & ((out['close'] - out['last_swing_high']) >= (min_break_atr * out['atr'])) # filter out weak breaks that don't exceed ATR threshold
    )
    # downward break detection
    down_break = (
        (out['close'] < out['last_swing_low']) # close below last swing low
        & (out['close'].shift(1) >= out['last_swing_low']) # Previous close at or above last swing low
        & ((out['last_swing_low'] - out['close']) >= (min_break_atr * out['atr'])) # filter out weak breaks that don't exceed ATR threshold
    )

    # Trend-state classification: 
    # BOS (Break of Structure) - Break in the same direction as the current trend, indicating continuation
    # CHOCH (Change of Character) - Break in the opposite direction of the current trend, indicating potential reversal
    trend = 0  # 1 up, -1 down, 0 unknown
    bullish_bos, bullish_choch, bearish_bos, bearish_choch = [], [], [], []

    for i in range(len(out)):
        upward_break_detected = bool(up_break.iloc[i]) if pd.notna(up_break.iloc[i]) else False
        downward_break_detected = bool(down_break.iloc[i]) if pd.notna(down_break.iloc[i]) else False

        if upward_break_detected and trend >= 0:
            bullish_bos.append(True); 
            bullish_choch.append(False); 
            bearish_bos.append(False); 
            bearish_choch.append(False)
            trend = 1
        elif upward_break_detected and trend < 0:
            bullish_bos.append(False); 
            bullish_choch.append(True); 
            bearish_bos.append(False); 
            bearish_choch.append(False)
            trend = 1
        elif downward_break_detected and trend <= 0:
            bullish_bos.append(False); 
            bullish_choch.append(False); 
            bearish_bos.append(True); 
            bearish_choch.append(False)
            trend = -1
        elif downward_break_detected and trend > 0:
            bullish_bos.append(False); 
            bullish_choch.append(False); 
            bearish_bos.append(False); 
            bearish_choch.append(True)
            trend = -1
        else:
            bullish_bos.append(False); 
            bullish_choch.append(False); 
            bearish_bos.append(False); 
            bearish_choch.append(False)

    out['bullish_bos'] = bullish_bos
    out['bullish_choch'] = bullish_choch
    out['bearish_bos'] = bearish_bos
    out['bearish_choch'] = bearish_choch

    out['bullish_shift'] = out['bullish_bos'] | out['bullish_choch']
    out['bearish_shift'] = out['bearish_bos'] | out['bearish_choch']

    return out


def prepare_ict_features(
    df,
    vol_look_back=50,
    require_volume=True,
    structure_left=3,
    structure_right=3,
    atr_len=14,
    min_break_atr=0.10,
    use_structure_filter=True,  # NEW
):
    """Prepare low timeframe (LTF) confirmation features from 1m OHLCV."""
    out = df.copy().sort_values('timestamp').reset_index(drop=True)
    out['timestamp'] = pd.to_datetime(out['timestamp'], utc=True)

    # Volume participation
    out['vol_sma'] = out['volume'].rolling(vol_look_back, min_periods=vol_look_back).mean()
    out['high_participation'] = out['volume'] > out['vol_sma']

    if use_structure_filter:
        # LTF BOS/CHOCH structure model
        structure = add_bos_choch(
            out,
            left=structure_left,
            right=structure_right,
            atr_len=atr_len,
            min_break_atr=min_break_atr,
        )

        cols_to_copy = [
            'atr',
            'swing_high_confirmed', 'swing_low_confirmed',
            'swing_high_price', 'swing_low_price',
            'last_swing_high', 'last_swing_low',
            'bullish_bos', 'bullish_choch', 'bearish_bos', 'bearish_choch',
            'bullish_shift', 'bearish_shift',
        ]
        out[cols_to_copy] = structure[cols_to_copy]
    else:
        # bypass structure filter, keep ATR for SL logic
        prev_close = out['close'].shift(1)
        tr = pd.concat([
            (out['high'] - out['low']).abs(),
            (out['high'] - prev_close).abs(),
            (out['low'] - prev_close).abs(),
        ], axis=1).max(axis=1)
        out['atr'] = tr.rolling(atr_len, min_periods=atr_len).mean()

        # placeholders so downstream columns always exist
        out['swing_high_confirmed'] = False
        out['swing_low_confirmed'] = False
        out['swing_high_price'] = np.nan
        out['swing_low_price'] = np.nan
        out['last_swing_high'] = np.nan
        out['last_swing_low'] = np.nan
        out['bullish_bos'] = False
        out['bullish_choch'] = False
        out['bearish_bos'] = False
        out['bearish_choch'] = False

        # force confirmation gate open
        out['bullish_shift'] = True
        out['bearish_shift'] = True

    if require_volume:
        out['bullish_shift_ok'] = out['bullish_shift'] & out['high_participation']
        out['bearish_shift_ok'] = out['bearish_shift'] & out['high_participation']
    else:
        out['bullish_shift_ok'] = out['bullish_shift']
        out['bearish_shift_ok'] = out['bearish_shift']

    # Continuation confluence proxies
    out['bullish_fvg'] = out['low'] > out['high'].shift(2)
    out['bearish_fvg'] = out['high'] < out['low'].shift(2)

    out['bullish_ob'] = (
        (out['close'].shift(1) < out['open'].shift(1))
        & (out['close'] > out['open'])
        & (out['close'] > out['high'].shift(1))
    )
    out['bearish_ob'] = (
        (out['close'].shift(1) > out['open'].shift(1))
        & (out['close'] < out['open'])
        & (out['close'] < out['low'].shift(1))
    )

    out['bullish_confluence'] = out['bullish_fvg'] | out['bullish_ob']
    out['bearish_confluence'] = out['bearish_fvg'] | out['bearish_ob']

    if require_volume:
        out['bullish_confluence_ok'] = out['bullish_confluence'] & out['high_participation']
        out['bearish_confluence_ok'] = out['bearish_confluence'] & out['high_participation']
    else:
        out['bullish_confluence_ok'] = out['bullish_confluence']
        out['bearish_confluence_ok'] = out['bearish_confluence']

    return out

In [ ]:
# Build per-symbol causal HTF context and LTF features
USE_VOLUME_FILTER = False
USE_STRUCTURE_FILTER = True
SWEEP_RELIABILITY_MIN = 61.0
REQUIRE_SWEEP_RECLAIM = True
DEDUPE_SWEEPS = True
SWEEP_PRICE_TOL_BPS = 2.0  # consider same-event sweeps within +/-2 bps at same timestamp+direction
symbol_contexts = {}


def dedupe_sweeps_same_event(sweeps_df: pd.DataFrame, price_tol_bps: float = 2.0) -> pd.DataFrame:
    if sweeps_df is None or sweeps_df.empty:
        return sweeps_df.copy()

    tol = float(price_tol_bps) / 10000.0
    work = sweeps_df.copy().sort_values(['timestamp', 'direction', 'level_price']).reset_index(drop=True)
    
    out_rows = []
    grouped = work.groupby(['timestamp', 'direction'], sort=False, dropna=False)

    for (_, _), grp in grouped:
        grp = grp.sort_values('level_price').reset_index(drop=True)
        current_cluster = []

        def flush_cluster(cluster_rows):
            if not cluster_rows:
                return
            cluster_df = pd.DataFrame(cluster_rows)
            if cluster_df['reliability_score'].notna().any():
                rep = cluster_df.loc[cluster_df['reliability_score'].idxmax()].copy()
            else:
                rep = cluster_df.iloc[0].copy()
            merged_types = sorted(cluster_df['level_type'].astype(str).unique().tolist())
            rep['merged_level_types'] = '|'.join(merged_types)
            rep['merged_events'] = int(len(cluster_df))
            
            # Boost score for confluence: each additional unique level type adds 5 points, capped at 100
            n_confluent = len(merged_types) - 1
            if n_confluent:
                confluence_boost = n_confluent * 5.0
                rep['reliability_score'] = float(np.clip(rep['reliability_score'] + confluence_boost, 0.0, 100.0))
                rep['confluence_boost'] = confluence_boost
            else:
                rep['confluence_boost'] = 0.0
                
            out_rows.append(rep)

        for _, row in grp.iterrows():
            row_dict = row.to_dict()
            if not current_cluster:
                current_cluster.append(row_dict)
                continue

            anchor_price = float(current_cluster[0]['level_price'])  # compare to first, not last
            this_price = float(row_dict['level_price'])
            price_ref = max(abs(anchor_price), abs(this_price), 1e-12)
            is_same_cluster = abs(this_price - anchor_price) <= (price_ref * tol)

            if is_same_cluster:
                current_cluster.append(row_dict)
            else:
                flush_cluster(current_cluster)
                current_cluster = [row_dict]

        flush_cluster(current_cluster)

    deduped = pd.DataFrame(out_rows).sort_values(['timestamp', 'direction', 'level_price']).reset_index(drop=True)
    return deduped

In [ ]:
for symbol, bars in market_bars.items():
    df_15m_ctx_sym = add_ict_liquidity_pools(bars["df_15m"])

    levels_df_sym, sweeps_df_sym_all = build_levels_and_sweeps_causal(df_15m_ctx_sym)

    sweeps_df_sym = sweeps_df_sym_all.copy()
    
    sweeps_pre_dedupe = len(sweeps_df_sym)
    if DEDUPE_SWEEPS and not sweeps_df_sym.empty:
        sweeps_df_sym = dedupe_sweeps_same_event(sweeps_df_sym, price_tol_bps=SWEEP_PRICE_TOL_BPS)
        
    if not sweeps_df_sym.empty:
        # Filter AFTER dedupe so confluence boost can save borderline sweeps
        sweeps_df_sym = sweeps_df_sym[sweeps_df_sym["reliability_score"] >= SWEEP_RELIABILITY_MIN]

    sweeps_df_sym = sweeps_df_sym.reset_index(drop=True)
    sweeps_removed_dedupe = sweeps_pre_dedupe - len(sweeps_df_sym)

    df_ltf_sym = prepare_ict_features(
        bars["df_1m"],
        vol_look_back=50,
        require_volume=USE_VOLUME_FILTER,
        structure_left=3, # longer look-back/look-forward for more reliable structure detection, since LTF features are used for confirmation after HTF sweeps
        structure_right=3, # longer look-back/look-forward for more reliable structure detection, since LTF features are used for confirmation after HTF sweeps
        atr_len=14,
        min_break_atr=0.10,
        use_structure_filter=USE_STRUCTURE_FILTER
    )

    symbol_contexts[symbol] = {
        "df_1m": bars["df_1m"],
        "df_15m": bars["df_15m"],
        "df_15m_ctx": df_15m_ctx_sym,
        "df_ltf": df_ltf_sym,
        "levels_df": levels_df_sym,
        "sweeps_df": sweeps_df_sym,
        "sweeps_df_all": sweeps_df_sym_all,
        "sweeps_pre_dedupe": sweeps_pre_dedupe,
        "sweeps_removed_dedupe": sweeps_removed_dedupe,
    }

# Keep single-symbol aliases for downstream chart/debug cells
if SYMBOL in symbol_contexts:
    df_1m = symbol_contexts[SYMBOL]["df_1m"]
    df_15m = symbol_contexts[SYMBOL]["df_15m"]
    df_15m_ctx = symbol_contexts[SYMBOL]["df_15m_ctx"]
    df_ltf = symbol_contexts[SYMBOL]["df_ltf"]
    levels_df = symbol_contexts[SYMBOL]["levels_df"]
    sweeps_df = symbol_contexts[SYMBOL]["sweeps_df"]
    df = df_ltf

In [ ]:
print(f"Symbols in run: {', '.join(symbol_contexts.keys())}")
print(f"Volume filter enabled: {USE_VOLUME_FILTER}")
print(f"Sweep reliability min score: {SWEEP_RELIABILITY_MIN:.1f}")
print(f"Require sweep reclaim: {REQUIRE_SWEEP_RECLAIM}")
print(f"Sweep dedup enabled: {DEDUPE_SWEEPS} (tol={SWEEP_PRICE_TOL_BPS:.1f} bps)")

frames = [
    ctx["sweeps_df"].assign(symbol=symbol)
    for symbol, ctx in symbol_contexts.items()
    if not ctx["sweeps_df"].empty
]

all_post_dedupe_eligible = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if all_post_dedupe_eligible.empty:
    print("No post-dedupe eligible sweeps found.")
else:
    s = all_post_dedupe_eligible["reliability_score"]

    all_post_dedupe_eligible["score_bucket"] = pd.Categorical(
        np.where(s < 65, "<65", np.where(s < 80, "65-79", "80+")),
        categories=["<65", "65-79", "80+"],
        ordered=True,
    )

    bucket_counts = (
        all_post_dedupe_eligible["score_bucket"]
        .value_counts(sort=False)
        .rename_axis("score_bucket")
        .reset_index(name="count")
    )

    print(f"Sweep reliability min score: {SWEEP_RELIABILITY_MIN:.1f}")
    print(f"Total post-dedupe eligible sweeps: {len(all_post_dedupe_eligible)}")
    print(bucket_counts.to_string(index=False))

    print("\nBy symbol and bucket:")
    by_symbol_bucket = (
        all_post_dedupe_eligible
        .groupby(["symbol", "score_bucket"], observed=False)
        .size()
        .reset_index(name="count")
        .sort_values(["symbol", "score_bucket"])
    )
    print(by_symbol_bucket.to_string(index=False))

for symbol, ctx in symbol_contexts.items():
    print("\n" + "-" * 70)
    print(symbol)
    print(f"HTF bars (15m): {len(ctx['df_15m_ctx']):,}")
    print(f"LTF bars (1m):  {len(ctx['df_ltf']):,}")
    print(f"Key levels created: {len(ctx['levels_df']):,}")

    total_sweeps = len(ctx.get('sweeps_df_all', ctx['sweeps_df']))
    sweeps_pre_dedupe = int(ctx.get('sweeps_pre_dedupe', len(ctx['sweeps_df'])))
    eligible_sweeps = len(ctx['sweeps_df'])
    dedupe_removed = int(ctx.get('sweeps_removed_dedupe', 0))

    print(f"Sweeps detected (all):            {total_sweeps:,}")
    print(f"Sweeps eligible (pre-dedupe):     {sweeps_pre_dedupe:,}")
    print(f"Sweeps removed by dedupe:         {dedupe_removed:,}")
    print(f"Sweeps eligible (post-dedupe):    {eligible_sweeps:,}")

    if not ctx['sweeps_df'].empty and 'reliability_score' in ctx['sweeps_df'].columns:
        print(
            f"Eligible score stats: min={ctx['sweeps_df']['reliability_score'].min():.1f}, "
            f"median={ctx['sweeps_df']['reliability_score'].median():.1f}, "
            f"max={ctx['sweeps_df']['reliability_score'].max():.1f}"
        )

    if not ctx['levels_df'].empty:
        print("\nKey level counts by type:")
        print(ctx['levels_df'].groupby(['level_type', 'side']).size().to_string())

    if not ctx['sweeps_df'].empty:
        print("\nMost recent eligible sweeps:")
        cols = [
            'timestamp', 'direction', 'level_type', 'merged_level_types', 'level_price',
            'reliability_score', 'merged_events', 'pre_sweep_touches', 'wick_excess_atr', 'volume_spike_z'
        ]
        show_cols = [c for c in cols if c in ctx['sweeps_df'].columns]
        print(ctx['sweeps_df'][show_cols].tail(5).to_string(index=False))

# 3. ICT-STYLE BACKTEST (HTF sweep -> LTF confirmation)

In [ ]:
INITIAL_BALANCE = 50000
LEVERAGE = 5
RISK_PER_TRADE = 0.02
MIN_RR = 2
OTE_MIN = 0.61
OTE_MAX = 0.79
SETUP_TIMEOUT_MIN = 120 # 75  # 5 x 15m bars
MAINT_MARGIN_RATE = 0.005
MIN_NOTIONAL = 100.0

balance = INITIAL_BALANCE
active_setups = []
open_positions = []
trades = []
symbol_realized_pnl = {symbol: 0.0 for symbol in symbol_contexts.keys()}

# Pre-index LTF by timestamp for causal per-timestamp processing
ltf_by_symbol = {}
all_timestamps = set()
for symbol, ctx in symbol_contexts.items():
    ltf = ctx['df_ltf'].copy()
    ltf['timestamp'] = pd.to_datetime(ltf['timestamp'], utc=True)
    ltf_idx = ltf.set_index('timestamp', drop=False).sort_index()
    ltf_by_symbol[symbol] = ltf_idx
    all_timestamps.update(ltf_idx.index.tolist())

timeline = sorted(all_timestamps)
last_close_by_symbol = {}

# Trigger setups only after corresponding HTF bar is fully closed
sweeps_by_trigger_ts = {}
for symbol, ctx in symbol_contexts.items():
    sweeps_df = ctx['sweeps_df']
    if sweeps_df.empty:
        continue
    sweeps_tmp = sweeps_df.copy()
    sweeps_tmp['timestamp'] = pd.to_datetime(sweeps_tmp['timestamp'], utc=True)
    sweeps_tmp['trigger_time'] = sweeps_tmp['timestamp'] + pd.Timedelta(minutes=15)
    for _, ev in sweeps_tmp.iterrows():
        event = ev.to_dict()
        event['symbol'] = symbol
        sweeps_by_trigger_ts.setdefault(event['trigger_time'], []).append(event)

In [ ]:
def get_last_close(now_ts, ltf_by_symbol, last_close_by_symbol=None):
    if last_close_by_symbol is None:
        last_close_by_symbol = {}
        
    for symbol, ltf_idx in ltf_by_symbol.items():
        if now_ts in ltf_idx.index:
            last_close_by_symbol[symbol] = float(ltf_idx.loc[now_ts, "close"])
    return last_close_by_symbol

def release_setups(now_ts, sweeps_by_trigger_ts, active_setups):
    if now_ts not in sweeps_by_trigger_ts:
        return active_setups
    for ev in sweeps_by_trigger_ts[now_ts]:
        active_setups.append({
            "symbol": ev["symbol"],
            "direction": ev["direction"],
            "sweep_time": ev["timestamp"],
            "trigger_time": ev["trigger_time"],
            "sweep_price": ev["level_price"],
            "sweep_level_type": ev["level_type"],
            "confirmed": False,
            "ote_close_ok": False,
        })
        
    return active_setups

In [ ]:
def confirm_or_timeout(now_ts, active_setups, ltf_by_symbol):
    updated = []
    for s in active_setups:
        age_min = (now_ts - pd.Timestamp(s["trigger_time"])).total_seconds() / 60.0
        if age_min > SETUP_TIMEOUT_MIN:
            continue

        # check for confirmation conditions in LTF features at the current timestamp
        # if not available or displacement is not positive, keep waiting until timeout
        symbol = s["symbol"]
        current = ltf_by_symbol[symbol].loc[now_ts] if now_ts in ltf_by_symbol[symbol].index else None
        if current is None:
            updated.append(s)
            continue

        if not s["confirmed"]:
            if s["direction"] == "SHORT" and bool(current["bearish_shift_ok"]):
                displacement = float(s["sweep_price"]) - float(current["low"])
                if displacement > 0:
                    s["confirmed"] = True
                    s["ote_close_ok"] = False
                    s["ote_low"] = float(current["low"]) + displacement * OTE_MIN
                    s["ote_high"] = float(current["low"]) + displacement * OTE_MAX

            elif s["direction"] == "LONG" and bool(current["bullish_shift_ok"]):
                displacement = float(current["high"]) - float(s["sweep_price"])
                if displacement > 0:
                    s["confirmed"] = True
                    s["ote_close_ok"] = False
                    s["ote_low"] = float(current["high"]) - displacement * OTE_MAX
                    s["ote_high"] = float(current["high"]) - displacement * OTE_MIN

        updated.append(s)
    return updated

def ote_gate(now_ts, active_setups, ltf_by_symbol):
    for s in active_setups:
        if not (s["confirmed"] and not s["ote_close_ok"]):
            continue

        symbol = s["symbol"]
        current = ltf_by_symbol[symbol].loc[now_ts] if now_ts in ltf_by_symbol[symbol].index else None
        if current is None:
            continue
        if isinstance(current, pd.DataFrame):
            current = current.iloc[-1]

        close_in_ote = s["ote_low"] <= float(current["close"]) <= s["ote_high"]
        if close_in_ote:
            s["ote_close_ok"] = True
            s["ote_pass_time"] = now_ts
            if s["direction"] == "SHORT":
                s["ote_pass_confluence_ok"] = bool(current["bearish_confluence_ok"])
                s["ote_pass_rejection_ok"] = float(current["close"]) < float(current["open"])
            else:
                s["ote_pass_confluence_ok"] = bool(current["bullish_confluence_ok"])
                s["ote_pass_rejection_ok"] = float(current["close"]) > float(current["open"])
            
    return active_setups

In [ ]:
POST_OTE_ENTRY_TIMEOUT_MIN = 30  # lifetime after OTE pass
def calculate_structural_stop_and_rr(current, setup, entry, tp):
    atr = float(current["atr"]) if pd.notna(current.get("atr", np.nan)) else 0.0
    atr_buffer = max(atr * 0.20, entry * 0.0005)  # 20% ATR or 5 bps

    if setup["direction"] == "SHORT":
        structural_sl = max(
            float(current["high"]),
            float(setup["sweep_price"]),
            float(setup["ote_high"]),
        ) + atr_buffer

        if structural_sl <= entry or float(tp) >= entry:
            return np.nan, np.nan

        rr = (entry - float(tp)) / (structural_sl - entry)
        return structural_sl, rr

    structural_sl = min(
        float(current["low"]),
        float(setup["sweep_price"]),
        float(setup["ote_low"]),
    ) - atr_buffer

    if structural_sl >= entry or float(tp) <= entry:
        return np.nan, np.nan

    rr = (float(tp) - entry) / (entry - structural_sl)
    return structural_sl, rr


def try_entries(now_ts, active_setups, ltf_by_symbol,
                symbol_contexts, open_positions, balance,
                last_close_by_symbol):
    still_pending = []

    for s in active_setups:
        entered = False
        symbol = s["symbol"]
        current = ltf_by_symbol[symbol].loc[now_ts] if now_ts in ltf_by_symbol[symbol].index else None
        if current is None:
            still_pending.append(s)
            continue
        if isinstance(current, pd.DataFrame):
            current = current.iloc[-1]

        if s.get("ote_close_ok") and pd.notna(s.get("ote_pass_time", pd.NaT)):
            post_ote_age_min = (now_ts - pd.Timestamp(s["ote_pass_time"])).total_seconds() / 60.0
            if post_ote_age_min > POST_OTE_ENTRY_TIMEOUT_MIN:
                continue

        equity = balance  # + unrealized_pnl
        used_margin = sum(p["margin_req"] for p in open_positions)
        free_margin = max(0.0, equity - used_margin)

        if s["confirmed"] and s["ote_close_ok"] and free_margin > 0:
            levels_df = symbol_contexts[symbol]["levels_df"]

            if s["direction"] == "SHORT":
                in_zone = (float(current["low"]) <= s["ote_high"]) and (float(current["high"]) >= s["ote_low"])
                confluence = bool(current["bearish_confluence_ok"])
                rejection = float(current["close"]) < float(current["open"])

                if in_zone and confluence and rejection:
                    entry = float(current["close"])
                    tp = nearest_key_level_live(levels_df, now_ts, entry, "SHORT")

                    if pd.notna(tp) and entry > float(tp):
                        sl, rr = calculate_structural_stop_and_rr(current, s, entry, tp)
                        #tp_dist = entry - float(tp)
                        #max_sl_dist = tp_dist / MIN_RR
                        #max_sl = entry + max_sl_dist
                        #ote_high = s["ote_high"]
                        #ote_low = s["ote_low"]
                        #min_sl = ote_high + (ote_high - ote_low) * 0.25 if pd.notna(ote_low) else ote_high * 1.001
                        #sl = max(max_sl, min_sl)

                        if pd.notna(sl) and pd.notna(rr) and rr >= MIN_RR:
                                risk_amount = equity * RISK_PER_TRADE
                                sl_distance = sl - entry
                                qty = min(risk_amount / sl_distance, (free_margin * LEVERAGE) / entry)

                                if qty > 0:
                                    notional = qty * entry
                                    margin_req = notional / LEVERAGE
                                    liq_price = entry * (1.0 + (1.0 / LEVERAGE - MAINT_MARGIN_RATE))
                                    risk_at_sl = sl_distance * qty

                                    if (
                                        notional >= MIN_NOTIONAL
                                        and sl < liq_price
                                        and margin_req <= free_margin
                                        and risk_at_sl <= risk_amount
                                    ):
                                        open_positions.append({
                                            "symbol": symbol,
                                            "type": "SHORT",
                                            "entry": entry,
                                            "sl": sl,
                                            "tp": float(tp),
                                            "qty": qty,
                                            "notional": notional,
                                            "margin_req": margin_req,
                                            "liq_price": liq_price,
                                            "entry_time": now_ts,
                                            "entry_equity": equity,
                                            "risk_budget": risk_amount,
                                            "risk_at_sl": risk_at_sl,
                                            "rr_at_entry": rr,
                                            "source_sweep_time": s["sweep_time"],
                                            "source_level_type": s["sweep_level_type"],
                                        })
                                        entered = True

            else:
                in_zone = (float(current["low"]) <= s["ote_high"]) and (float(current["high"]) >= s["ote_low"])
                confluence = bool(current["bullish_confluence_ok"])
                rejection = float(current["close"]) > float(current["open"])

                if in_zone and confluence and rejection:
                    entry = float(current["close"])
                    tp = nearest_key_level_live(levels_df, now_ts, entry, "LONG")

                    if pd.notna(tp) and float(tp) > entry:
                        sl, rr = calculate_structural_stop_and_rr(current, s, entry, tp)
    
                        #tp_dist = float(tp) - entry
                        #max_sl_dist = tp_dist / MIN_RR
                        #max_sl = entry - max_sl_dist
                        #ote_high = s["ote_high"]
                        #ote_low = s["ote_low"]
                        #max_sl_ote = (
                        #    ote_low - (ote_high - ote_low) * 0.25
                        #    if pd.notna(ote_high) and pd.notna(ote_low)
                        #    else ote_low * 0.999
                        #)
                        #sl = min(max_sl, max_sl_ote)
                        if pd.notna(sl) and pd.notna(rr) and rr >= MIN_RR:
                                risk_amount = equity * RISK_PER_TRADE
                                sl_distance = entry - sl
                                qty = min(risk_amount / sl_distance, (free_margin * LEVERAGE) / entry)

                                if qty > 0:
                                    notional = qty * entry
                                    margin_req = notional / LEVERAGE
                                    liq_price = entry * (1.0 - (1.0 / LEVERAGE - MAINT_MARGIN_RATE))
                                    risk_at_sl = sl_distance * qty

                                    if (
                                        notional >= MIN_NOTIONAL
                                        and sl > liq_price
                                        and margin_req <= free_margin
                                        and risk_at_sl <= risk_amount
                                    ):
                                        open_positions.append({
                                            "symbol": symbol,
                                            "type": "LONG",
                                            "entry": entry,
                                            "sl": sl,
                                            "tp": float(tp),
                                            "qty": qty,
                                            "notional": notional,
                                            "margin_req": margin_req,
                                            "liq_price": liq_price,
                                            "entry_time": now_ts,
                                            "entry_equity": equity,
                                            "risk_budget": risk_amount,
                                            "risk_at_sl": risk_at_sl,
                                            "rr_at_entry": rr,
                                            "source_sweep_time": s["sweep_time"],
                                            "source_level_type": s["sweep_level_type"],
                                        })
                                        entered = True

        if not entered:
            still_pending.append(s)

    return still_pending, open_positions

In [ ]:
def manage_positions(now_ts, open_positions, ltf_by_symbol, balance, symbol_realized_pnl, trades):
    remaining = []

    for pos in open_positions:
        symbol = pos["symbol"]
        current = ltf_by_symbol[symbol].loc[now_ts] if now_ts in ltf_by_symbol[symbol].index else None
        if current is None:
            remaining.append(pos)
            continue
        if isinstance(current, pd.DataFrame):
            current = current.iloc[-1]

        if pd.Timestamp(pos["entry_time"]) == pd.Timestamp(now_ts):
            remaining.append(pos)
            continue

        exit_reason, pnl = None, 0.0

        if pos["type"] == "SHORT":
            hit_liq = float(current["high"]) >= pos["liq_price"]
            hit_sl  = float(current["high"]) >= pos["sl"]
            hit_tp  = float(current["low"]) <= pos["tp"]

            if hit_liq or hit_sl:
                if hit_liq and hit_sl and min(pos["liq_price"], pos["sl"]) == pos["liq_price"]:
                    exit_reason, pnl = "LIQ", -pos["margin_req"]
                elif hit_liq:
                    exit_reason, pnl = "LIQ", -pos["margin_req"]
                else:
                    exit_reason, pnl = "SL", (pos["entry"] - pos["sl"]) * pos["qty"]
            elif hit_tp:
                exit_reason, pnl = "TP", (pos["entry"] - pos["tp"]) * pos["qty"]

        else:
            hit_liq = float(current["low"]) <= pos["liq_price"]
            hit_sl  = float(current["low"]) <= pos["sl"]
            hit_tp  = float(current["high"]) >= pos["tp"]

            if hit_liq or hit_sl:
                if hit_liq and hit_sl and max(pos["liq_price"], pos["sl"]) == pos["liq_price"]:
                    exit_reason, pnl = "LIQ", -pos["margin_req"]
                elif hit_liq:
                    exit_reason, pnl = "LIQ", -pos["margin_req"]
                else:
                    exit_reason, pnl = "SL", (pos["sl"] - pos["entry"]) * pos["qty"]
            elif hit_tp:
                exit_reason, pnl = "TP", (pos["tp"] - pos["entry"]) * pos["qty"]

        if exit_reason is None:
            remaining.append(pos)
            continue

        balance += pnl
        symbol_realized_pnl[symbol] = symbol_realized_pnl.get(symbol, 0.0) + pnl
        trades.append({
            "symbol": symbol,
            "entry_time": pos["entry_time"],
            "exit_time": now_ts,
            "type": pos["type"],
            "entry": pos["entry"],
            "sl": pos["sl"],
            "tp": pos["tp"],
            "liq_price": pos["liq_price"],
            "qty": pos["qty"],
            "notional": pos["notional"],
            "margin_req": pos["margin_req"],
            "entry_equity": pos.get("entry_equity", np.nan),
            "risk_budget": pos.get("risk_budget", np.nan),
            "risk_at_sl": pos.get("risk_at_sl", np.nan),
            "rr_at_entry": pos.get("rr_at_entry", np.nan),
            "exit_reason": exit_reason,
            "close_price": pos["tp"] if exit_reason == "TP" else (pos["sl"] if exit_reason == "SL" else pos["liq_price"]),
            "pnl": pnl,
            "balance": balance,
            "source_sweep_time": pos["source_sweep_time"],
            "source_level_type": pos["source_level_type"],
        })
    return remaining, balance, symbol_realized_pnl, trades


In [ ]:
last_close_by_symbol = None
active_setups = []
for current_time in timeline:
    # get closes for all symbols at current_time
    last_close_by_symbol = get_last_close(current_time, ltf_by_symbol, last_close_by_symbol)
    
    # check for new setups triggered by sweeps that occurred 15m ago at the current_time
    active_setups = release_setups(current_time, sweeps_by_trigger_ts, active_setups)
    
    # check for confirmation or timeout of active setups based on LTF features at the current_time
    active_setups = confirm_or_timeout(current_time, active_setups, ltf_by_symbol)
    
    # check for OTE gate conditions of active confirmed setups at the current_time
    active_setups = ote_gate(current_time, active_setups, ltf_by_symbol)
    
    # attempt entries for active setups that have passed OTE gate at the current_time, and update open positions and balance accordingly
    active_setups, open_positions = try_entries(current_time, active_setups, ltf_by_symbol, 
                                                symbol_contexts, open_positions, balance, last_close_by_symbol)
    
    # check for SL/TP/LIQ hits for open positions at the current_time, update balance and record trades accordingly
    open_positions, balance, symbol_realized_pnl, trades = manage_positions(
        current_time, open_positions, ltf_by_symbol, balance, symbol_realized_pnl, trades)

## Old Backtestin Loop (for reference)

In [ ]:
for now_ts in timeline:
    for symbol, ltf_idx in ltf_by_symbol.items():
        if now_ts in ltf_idx.index:
            row_now = ltf_idx.loc[now_ts]
            if isinstance(row_now, pd.DataFrame):
                row_now = row_now.iloc[-1]
            last_close_by_symbol[symbol] = float(row_now['close'])

    # 1) New setups become visible only at trigger_time
    if now_ts in sweeps_by_trigger_ts:
        for ev in sweeps_by_trigger_ts[now_ts]:
            assert now_ts == ev['trigger_time'], f"Event trigger time {ev['trigger_time']} does not match current time {now_ts}"
            active_setups.append({
                'symbol': ev['symbol'],
                'direction': ev['direction'],
                'sweep_time': ev['timestamp'],
                'trigger_time': ev['trigger_time'],
                'sweep_price': ev['level_price'],
                'sweep_level_type': ev['level_type'],
                'confirmed': False,
                'ote_close_ok': False,
            })

    # 2) Confirm setups and timeout old setups
    updated_setups = []
    for s in active_setups:
        age_min = (now_ts - pd.Timestamp(s['trigger_time'])).total_seconds() / 60.0
        if age_min > SETUP_TIMEOUT_MIN:
            continue

        symbol = s['symbol']
        ltf_idx = ltf_by_symbol[symbol]
        if now_ts not in ltf_idx.index:
            updated_setups.append(s)
            continue

        current = ltf_idx.loc[now_ts]
        if isinstance(current, pd.DataFrame):
            current = current.iloc[-1]

        if not s['confirmed']:
            if s['direction'] == 'SHORT' and current['bearish_shift_ok']:
                displacement = s['sweep_price'] - current['low']
                if displacement > 0:
                    s['confirmed'] = True
                    s['ote_close_ok'] = False
                    s['ote_low'] = current['low'] + displacement * OTE_MIN
                    s['ote_high'] = current['low'] + displacement * OTE_MAX

            elif s['direction'] == 'LONG' and current['bullish_shift_ok']:
                displacement = current['high'] - s['sweep_price']
                if displacement > 0:
                    s['confirmed'] = True
                    s['ote_close_ok'] = False
                    s['ote_low'] = current['high'] - displacement * OTE_MAX
                    s['ote_high'] = current['high'] - displacement * OTE_MIN

        updated_setups.append(s)
    active_setups = updated_setups

    # 2b) OTE close gate
    for s in active_setups:
        if s['confirmed'] and not s['ote_close_ok']:
            symbol = s['symbol']
            ltf_idx = ltf_by_symbol[symbol]
            if now_ts not in ltf_idx.index:
                continue
            current = ltf_idx.loc[now_ts]
            if isinstance(current, pd.DataFrame):
                current = current.iloc[-1]

            if s['direction'] == 'SHORT' and current['close'] >= s['ote_high']:
                s['ote_close_ok'] = True
            elif s['direction'] == 'LONG' and current['close'] <= s['ote_low']:
                s['ote_close_ok'] = True

    # 3) Entry checks with shared portfolio equity/margin
    still_pending = []
    for s in active_setups:
        entered = False
        symbol = s['symbol']
        ltf_idx = ltf_by_symbol[symbol]

        if now_ts not in ltf_idx.index:
            still_pending.append(s)
            continue

        current = ltf_idx.loc[now_ts]
        if isinstance(current, pd.DataFrame):
            current = current.iloc[-1]

        unrealized_pnl = 0.0
        for p in open_positions:
            mark = last_close_by_symbol.get(p['symbol'], p['entry'])
            if p['type'] == 'SHORT':
                unrealized_pnl += (p['entry'] - mark) * p['qty']
            else:
                unrealized_pnl += (mark - p['entry']) * p['qty']

        equity = balance + unrealized_pnl
        used_margin = sum(p['margin_req'] for p in open_positions)
        free_margin = max(0.0, equity - used_margin)

        if s['confirmed'] and s['ote_close_ok'] and free_margin > 0:
            levels_df = symbol_contexts[symbol]['levels_df']

            if s['direction'] == 'SHORT':
                in_zone = (current['low'] <= s['ote_high']) and (current['high'] >= s['ote_low'])
                confluence = current['bearish_confluence_ok']
                rejection = current['close'] < current['open']

                if in_zone and confluence and rejection:
                    entry = float(current['close'])
                    sl = float(max(current['high'], s['sweep_price']) * 1.001)
                    tp = nearest_key_level_live(levels_df, now_ts, entry, 'SHORT')

                    if pd.notna(tp) and entry > tp and sl > entry:
                        rr = (entry - float(tp)) / (sl - entry)
                        if rr >= MIN_RR:
                            risk_amount = equity * RISK_PER_TRADE
                            sl_distance = sl - entry
                            risk_qty = risk_amount / sl_distance
                            max_qty_by_margin = (free_margin * LEVERAGE) / entry
                            qty = min(risk_qty, max_qty_by_margin)

                            if qty > 0:
                                notional = qty * entry
                                margin_req = notional / LEVERAGE
                                liq_price = entry * (1.0 + (1.0 / LEVERAGE - MAINT_MARGIN_RATE))
                                risk_at_sl = sl_distance * qty

                                if (
                                    notional >= MIN_NOTIONAL
                                    and sl < liq_price
                                    and margin_req <= free_margin
                                    and risk_at_sl <= risk_amount
                                ):
                                    open_positions.append({
                                        'symbol': symbol,
                                        'type': 'SHORT',
                                        'entry': entry,
                                        'sl': sl,
                                        'tp': float(tp),
                                        'qty': qty,
                                        'notional': notional,
                                        'margin_req': margin_req,
                                        'liq_price': liq_price,
                                        'entry_time': now_ts,
                                        'entry_equity': equity,
                                        'risk_budget': risk_amount,
                                        'risk_at_sl': risk_at_sl,
                                        'rr_at_entry': rr,
                                        'source_sweep_time': s['sweep_time'],
                                        'source_level_type': s['sweep_level_type'],
                                    })
                                    entered = True

            else:  # LONG
                in_zone = (current['low'] <= s['ote_high']) and (current['high'] >= s['ote_low'])
                confluence = current['bullish_confluence_ok']
                rejection = current['close'] > current['open']

                if in_zone and confluence and rejection:
                    entry = float(current['close'])
                    sl = float(min(current['low'], s['sweep_price']) * 0.999)
                    tp = nearest_key_level_live(levels_df, now_ts, entry, 'LONG')

                    if pd.notna(tp) and tp > entry and entry > sl:
                        rr = (float(tp) - entry) / (entry - sl)
                        if rr >= MIN_RR:
                            risk_amount = equity * RISK_PER_TRADE
                            sl_distance = entry - sl
                            risk_qty = risk_amount / sl_distance
                            max_qty_by_margin = (free_margin * LEVERAGE) / entry
                            qty = min(risk_qty, max_qty_by_margin)

                            if qty > 0:
                                notional = qty * entry
                                margin_req = notional / LEVERAGE
                                liq_price = entry * (1.0 - (1.0 / LEVERAGE - MAINT_MARGIN_RATE))
                                risk_at_sl = sl_distance * qty

                                if (
                                    notional >= MIN_NOTIONAL
                                    and sl > liq_price
                                    and margin_req <= free_margin
                                    and risk_at_sl <= risk_amount
                                ):
                                    open_positions.append({
                                        'symbol': symbol,
                                        'type': 'LONG',
                                        'entry': entry,
                                        'sl': sl,
                                        'tp': float(tp),
                                        'qty': qty,
                                        'notional': notional,
                                        'margin_req': margin_req,
                                        'liq_price': liq_price,
                                        'entry_time': now_ts,
                                        'entry_equity': equity,
                                        'risk_budget': risk_amount,
                                        'risk_at_sl': risk_at_sl,
                                        'rr_at_entry': rr,
                                        'source_sweep_time': s['sweep_time'],
                                        'source_level_type': s['sweep_level_type'],
                                    })
                                    entered = True

        if not entered:
            still_pending.append(s)

    active_setups = still_pending

    # 4) Manage open positions (per-symbol row at current timestamp)
    remaining_positions = []
    for pos in open_positions:
        symbol = pos['symbol']
        ltf_idx = ltf_by_symbol[symbol]

        if now_ts not in ltf_idx.index:
            remaining_positions.append(pos)
            continue

        current = ltf_idx.loc[now_ts]
        if isinstance(current, pd.DataFrame):
            current = current.iloc[-1]

        if pd.Timestamp(pos['entry_time']) == pd.Timestamp(now_ts):
            remaining_positions.append(pos)
            continue

        exit_reason = None
        pnl = 0.0

        if pos['type'] == 'SHORT':
            hit_liq = current['high'] >= pos['liq_price']
            hit_sl = current['high'] >= pos['sl']
            hit_tp = current['low'] <= pos['tp']

            if hit_liq or hit_sl:
                if hit_liq and hit_sl:
                    first_adverse = min(pos['liq_price'], pos['sl'])
                    if first_adverse == pos['liq_price']:
                        exit_reason = 'LIQ'
                        pnl = -pos['margin_req']
                    else:
                        exit_reason = 'SL'
                        pnl = (pos['entry'] - pos['sl']) * pos['qty']
                elif hit_liq:
                    exit_reason = 'LIQ'
                    pnl = -pos['margin_req']
                else:
                    exit_reason = 'SL'
                    pnl = (pos['entry'] - pos['sl']) * pos['qty']
            elif hit_tp:
                exit_reason = 'TP'
                pnl = (pos['entry'] - pos['tp']) * pos['qty']

        else:
            hit_liq = current['low'] <= pos['liq_price']
            hit_sl = current['low'] <= pos['sl']
            hit_tp = current['high'] >= pos['tp']

            if hit_liq or hit_sl:
                if hit_liq and hit_sl:
                    first_adverse = max(pos['liq_price'], pos['sl'])
                    if first_adverse == pos['liq_price']:
                        exit_reason = 'LIQ'
                        pnl = -pos['margin_req']
                    else:
                        exit_reason = 'SL'
                        pnl = (pos['sl'] - pos['entry']) * pos['qty']
                elif hit_liq:
                    exit_reason = 'LIQ'
                    pnl = -pos['margin_req']
                else:
                    exit_reason = 'SL'
                    pnl = (pos['sl'] - pos['entry']) * pos['qty']
            elif hit_tp:
                exit_reason = 'TP'
                pnl = (pos['tp'] - pos['entry']) * pos['qty']

        if exit_reason:
            balance += pnl
            symbol_realized_pnl[symbol] = symbol_realized_pnl.get(symbol, 0.0) + pnl
            trades.append({
                'symbol': symbol,
                'entry_time': pos['entry_time'],
                'exit_time': now_ts,
                'type': pos['type'],
                'entry': pos['entry'],
                'sl': pos['sl'],
                'tp': pos['tp'],
                'liq_price': pos['liq_price'],
                'qty': pos['qty'],
                'notional': pos['notional'],
                'margin_req': pos['margin_req'],
                'entry_equity': pos.get('entry_equity', np.nan),
                'risk_budget': pos.get('risk_budget', np.nan),
                'risk_at_sl': pos.get('risk_at_sl', np.nan),
                'rr_at_entry': pos.get('rr_at_entry', np.nan),
                'exit_reason': exit_reason,
                'pnl': pnl,
                'balance': balance,
                'source_sweep_time': pos['source_sweep_time'],
                'source_level_type': pos['source_level_type'],
            })
        else:
            remaining_positions.append(pos)

    open_positions = remaining_positions




## Results

In [ ]:
# Final MtM equity for shared portfolio
open_unrealized_pnl = 0.0
for p in open_positions:
    mark = last_close_by_symbol.get(p['symbol'], p['entry'])
    if p['type'] == 'SHORT':
        open_unrealized_pnl += (p['entry'] - mark) * p['qty']
    else:
        open_unrealized_pnl += (mark - p['entry']) * p['qty']

final_equity = balance + open_unrealized_pnl

trades_df = pd.DataFrame(trades)
if not trades_df.empty:
    trades_df['entry_time'] = pd.to_datetime(trades_df['entry_time'], utc=True)
    trades_df['exit_time'] = pd.to_datetime(trades_df['exit_time'], utc=True)
    trades_df['hold_minutes'] = ((trades_df['exit_time'] - trades_df['entry_time']).dt.total_seconds() / 60).astype(int)
    trades_df['close_price'] = np.where(
        trades_df['exit_reason'] == 'TP', trades_df['tp'],
        np.where(trades_df['exit_reason'] == 'SL', trades_df['sl'], trades_df['liq_price'])
    )

symbol_results = {
    symbol: {
        'symbol': symbol,
        'trades_df': trades_df[trades_df['symbol'] == symbol].copy() if not trades_df.empty else pd.DataFrame(),
        'realized_pnl': symbol_realized_pnl.get(symbol, 0.0),
    }
    for symbol in symbol_contexts.keys()
}

In [ ]:

    print(f"\nSymbols tested: {', '.join(symbol_contexts.keys())}")
    print(f"Initial Portfolio Balance: ${INITIAL_BALANCE:,.2f}")
    print(f"Final Balance (realized):  ${balance:,.2f}")
    print(f"Open Unrealized P&L:      ${open_unrealized_pnl:,.2f}")
    print(f"Final Equity (MtM):       ${final_equity:,.2f}")
    print(f"Net P&L (realized):       ${balance - INITIAL_BALANCE:,.2f}")
    print(f"Net P&L (MtM):            ${final_equity - INITIAL_BALANCE:,.2f}")
    print(f"Return (MtM):             {((final_equity / INITIAL_BALANCE) - 1) * 100:.2f}%")

    if trades_df.empty:
        print("\nNo closed trades executed.")
    else:
        summary_rows = []
        for symbol in symbol_contexts.keys():
            symbol_trades = trades_df[trades_df['symbol'] == symbol]
            wins = int((symbol_trades['pnl'] > 0).sum())
            losses = int((symbol_trades['pnl'] < 0).sum())
            win_rate = (wins / len(symbol_trades) * 100) if len(symbol_trades) > 0 else 0.0
            summary_rows.append({
                'symbol': symbol,
                'trades': len(symbol_trades),
                'wins': wins,
                'losses': losses,
                'win_rate_pct': win_rate,
                'realized_pnl': symbol_trades['pnl'].sum() if not symbol_trades.empty else 0.0,
            })

        summary_df = pd.DataFrame(summary_rows).sort_values('realized_pnl', ascending=False)

        print("\nPer-symbol summary (shared portfolio):")
        print(summary_df.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

        print(f"\nTotal closed trades (all symbols): {len(trades_df)}")
        all_win_rate = ((trades_df['pnl'] > 0).mean() * 100) if len(trades_df) else 0.0
        print(f"Overall win rate (all symbols):    {all_win_rate:.1f}%")
        print(f"Open positions (still live):       {len(open_positions)}")

        lifecycle_cols = [
            'symbol', 'entry_time', 'exit_time', 'hold_minutes', 'type',
            'entry', 'close_price', 'tp', 'sl', 'exit_reason',
            'pnl', 'balance', 'source_level_type'
        ]
        print("\nTrade lifecycle (all symbols):")
        print(trades_df[lifecycle_cols].to_string(index=False))

        diag_cols = [
            'symbol', 'entry_time', 'exit_time', 'hold_minutes',  'type', 'entry_equity', 'risk_budget', 'risk_at_sl',
            'rr_at_entry', 'qty', 'notional', 'margin_req', 'exit_reason', 'pnl'
        ]
        print("\nRisk diagnostics (all symbols):")
        print(trades_df[diag_cols].to_string(index=False))

In [ ]:
trades_df

In [ ]:
open_positions

In [ ]:
from collections import defaultdict

def build_symbol_funnel(symbol: str, ctx: dict):
    df_ltf = ctx['df_ltf'].copy().sort_values('timestamp').reset_index(drop=True)
    df_ltf['timestamp'] = pd.to_datetime(df_ltf['timestamp'], utc=True)
    ltf_idx = df_ltf.set_index('timestamp', drop=False).sort_index()

    sweeps_df = ctx['sweeps_df']
    sweeps_by_trigger_ts = {}
    if not sweeps_df.empty:
        sweeps_tmp = sweeps_df.copy()
        sweeps_tmp['timestamp'] = pd.to_datetime(sweeps_tmp['timestamp'], utc=True)
        sweeps_tmp['trigger_time'] = sweeps_tmp['timestamp'] + pd.Timedelta(minutes=15)
        for _, ev in sweeps_tmp.iterrows():
            sweeps_by_trigger_ts.setdefault(ev['trigger_time'], []).append(ev.to_dict())

    levels_df = ctx['levels_df']
    counters = defaultdict(int)
    active_setups = []

    # diagnostics proxy (funnel-only, not full portfolio path simulation)
    equity_proxy = float(INITIAL_BALANCE)
    free_margin_proxy = float(INITIAL_BALANCE)

    for now_ts in ltf_idx.index:
        current = ltf_idx.loc[now_ts]
        if isinstance(current, pd.DataFrame):
            current = current.iloc[-1]

        if now_ts in sweeps_by_trigger_ts:
            for ev in sweeps_by_trigger_ts[now_ts]:
                active_setups.append({
                    'direction': ev['direction'],
                    'sweep_time': ev['timestamp'],
                    'trigger_time': ev['trigger_time'],
                    'sweep_price': ev['level_price'],
                    'sweep_level_type': ev['level_type'],
                    'confirmed': False,
                    'ote_close_ok': False,
                })
            counters['sweeps'] += len(sweeps_by_trigger_ts[now_ts])

        updated_setups = []
        for s in active_setups:
            age_min = (now_ts - pd.Timestamp(s['trigger_time'])).total_seconds() / 60.0
            if age_min > SETUP_TIMEOUT_MIN:
                if not s['confirmed']:
                    counters['timeout_pre_confirm'] += 1
                elif not s['ote_close_ok']:
                    counters['timeout_pre_ote'] += 1
                else:
                    counters['timeout_post_ote'] += 1
                continue

            # 1) confirmation (new loop)
            if not s['confirmed']:
                if s['direction'] == 'SHORT' and bool(current['bearish_shift_ok']):
                    displacement = float(s['sweep_price']) - float(current['low'])
                    if displacement > 0:
                        s['confirmed'] = True
                        s['ote_close_ok'] = False
                        s['ote_low'] = float(current['low']) + displacement * OTE_MIN
                        s['ote_high'] = float(current['low']) + displacement * OTE_MAX
                        counters['confirmed'] += 1

                elif s['direction'] == 'LONG' and bool(current['bullish_shift_ok']):
                    displacement = float(current['high']) - float(s['sweep_price'])
                    if displacement > 0:
                        s['confirmed'] = True
                        s['ote_close_ok'] = False
                        s['ote_low'] = float(current['high']) - displacement * OTE_MAX
                        s['ote_high'] = float(current['high']) - displacement * OTE_MIN
                        counters['confirmed'] += 1

            # 2) OTE gate (new loop: close inside OTE band)
            if s.get('confirmed') and not s['ote_close_ok']:
                close_in_ote = s['ote_low'] <= float(current['close']) <= s['ote_high']
                if close_in_ote:
                    s['ote_close_ok'] = True
                    counters['ote_gate_pass'] += 1

            # 3) entry checks (new loop)
            entered = False
            if s.get('confirmed') and s['ote_close_ok']:
                counters['entry_windows'] += 1
                in_zone = (float(current['low']) <= s['ote_high']) and (float(current['high']) >= s['ote_low'])

                if s['direction'] == 'SHORT':
                    confluence = bool(current['bearish_confluence_ok'])
                    rejection = float(current['close']) < float(current['open'])

                    if not in_zone:
                        counters['rej_in_zone'] += 1
                    elif not confluence:
                        counters['rej_confluence'] += 1
                    elif not rejection:
                        counters['rej_rejection'] += 1
                    else:
                        entry = float(current['close'])
                        tp = nearest_key_level_live(levels_df, now_ts, entry, 'SHORT')

                        if pd.isna(tp) or not (entry > float(tp)):
                            counters['rej_tp_or_structure'] += 1
                        else:
                            tp_dist = entry - float(tp)
                            max_sl_dist = tp_dist / MIN_RR
                            max_sl = entry + max_sl_dist

                            ote_high = s['ote_high']
                            ote_low = s['ote_low']
                            min_sl = ote_high + (ote_high - ote_low) * 0.25 if pd.notna(ote_low) else ote_high * 1.001
                            sl = max(max_sl, min_sl)

                            if not (sl > entry):
                                counters['rej_tp_or_structure'] += 1
                            else:
                                rr = (entry - float(tp)) / (sl - entry)
                                if rr < MIN_RR:
                                    counters['rej_rr'] += 1
                                else:
                                    risk_amount = equity_proxy * RISK_PER_TRADE
                                    sl_distance = sl - entry
                                    qty = min(risk_amount / sl_distance, (free_margin_proxy * LEVERAGE) / entry) if sl_distance > 0 else 0.0

                                    if qty <= 0:
                                        counters['rej_margin_or_qty'] += 1
                                    else:
                                        notional = qty * entry
                                        margin_req = notional / LEVERAGE
                                        liq_price = entry * (1.0 + (1.0 / LEVERAGE - MAINT_MARGIN_RATE))
                                        risk_at_sl = sl_distance * qty

                                        if notional < MIN_NOTIONAL:
                                            counters['rej_notional'] += 1
                                        elif margin_req > free_margin_proxy:
                                            counters['rej_margin_or_qty'] += 1
                                        elif not (sl < liq_price):
                                            counters['rej_liq_guard'] += 1
                                        elif not (risk_at_sl <= risk_amount + 1e-12):
                                            counters['rej_risk_guard'] += 1
                                        else:
                                            counters['entry_signals'] += 1
                                            entered = True
                else:
                    confluence = bool(current['bullish_confluence_ok'])
                    rejection = float(current['close']) > float(current['open'])

                    if not in_zone:
                        counters['rej_in_zone'] += 1
                    elif not confluence:
                        counters['rej_confluence'] += 1
                    elif not rejection:
                        counters['rej_rejection'] += 1
                    else:
                        entry = float(current['close'])
                        tp = nearest_key_level_live(levels_df, now_ts, entry, 'LONG')

                        if pd.isna(tp) or not (float(tp) > entry):
                            counters['rej_tp_or_structure'] += 1
                        else:
                            tp_dist = float(tp) - entry
                            max_sl_dist = tp_dist / MIN_RR
                            max_sl = entry - max_sl_dist

                            ote_high = s['ote_high']
                            ote_low = s['ote_low']
                            max_sl_ote = (
                                ote_low - (ote_high - ote_low) * 0.25
                                if pd.notna(ote_high) and pd.notna(ote_low)
                                else ote_low * 0.999
                            )
                            sl = min(max_sl, max_sl_ote)

                            if not (entry > sl):
                                counters['rej_tp_or_structure'] += 1
                            else:
                                rr = (float(tp) - entry) / (entry - sl)
                                if rr < MIN_RR:
                                    counters['rej_rr'] += 1
                                else:
                                    risk_amount = equity_proxy * RISK_PER_TRADE
                                    sl_distance = entry - sl
                                    qty = min(risk_amount / sl_distance, (free_margin_proxy * LEVERAGE) / entry) if sl_distance > 0 else 0.0

                                    if qty <= 0:
                                        counters['rej_margin_or_qty'] += 1
                                    else:
                                        notional = qty * entry
                                        margin_req = notional / LEVERAGE
                                        liq_price = entry * (1.0 - (1.0 / LEVERAGE - MAINT_MARGIN_RATE))
                                        risk_at_sl = sl_distance * qty

                                        if notional < MIN_NOTIONAL:
                                            counters['rej_notional'] += 1
                                        elif margin_req > free_margin_proxy:
                                            counters['rej_margin_or_qty'] += 1
                                        elif not (sl > liq_price):
                                            counters['rej_liq_guard'] += 1
                                        elif not (risk_at_sl <= risk_amount + 1e-12):
                                            counters['rej_risk_guard'] += 1
                                        else:
                                            counters['entry_signals'] += 1
                                            entered = True

            if entered:
                counters['setup_entered'] += 1
            else:
                updated_setups.append(s)

        active_setups = updated_setups

    counters['active_unresolved_end'] = len(active_setups)
    return dict(counters)


funnel_rows = []
volume_rows = []
trade_rows = []

for symbol, ctx in symbol_contexts.items():
    funnel = build_symbol_funnel(symbol, ctx)
    row = {'symbol': symbol}
    row.update(funnel)
    row['sweeps_total_all'] = len(ctx.get('sweeps_df_all', ctx['sweeps_df']))
    row['sweeps_eligible_pre_dedupe'] = int(ctx.get('sweeps_pre_dedupe', len(ctx['sweeps_df'])))
    row['sweeps_removed_dedupe'] = int(ctx.get('sweeps_removed_dedupe', 0))
    row['sweeps_eligible'] = len(ctx['sweeps_df'])
    if not ctx['sweeps_df'].empty and 'reliability_score' in ctx['sweeps_df'].columns:
        row['sweep_score_mean'] = float(ctx['sweeps_df']['reliability_score'].mean())
        row['sweep_score_median'] = float(ctx['sweeps_df']['reliability_score'].median())
        row['score_ge_80'] = int((ctx['sweeps_df']['reliability_score'] >= 80).sum())
        row['score_65_79'] = int(((ctx['sweeps_df']['reliability_score'] >= 65) & (ctx['sweeps_df']['reliability_score'] < 80)).sum())
        row['score_lt_65'] = int((ctx['sweeps_df']['reliability_score'] < 65).sum())
    else:
        row['sweep_score_mean'] = np.nan
        row['sweep_score_median'] = np.nan
        row['score_ge_80'] = 0
        row['score_65_79'] = 0
        row['score_lt_65'] = 0
    funnel_rows.append(row)

    ltf = ctx['df_ltf'].copy()
    quote_vol = ltf['close'] * ltf['volume']
    volume_rows.append({
        'symbol': symbol,
        'median_base_volume_1m': float(ltf['volume'].median()),
        'median_quote_volume_1m': float(quote_vol.median()),
        'mean_quote_volume_1m': float(quote_vol.mean()),
        'high_participation_rate': float(ltf['high_participation'].mean()),
        'bullish_shift_ok_rate': float(ltf['bullish_shift_ok'].mean()),
        'bearish_shift_ok_rate': float(ltf['bearish_shift_ok'].mean()),
        'bullish_confluence_ok_rate': float(ltf['bullish_confluence_ok'].mean()),
        'bearish_confluence_ok_rate': float(ltf['bearish_confluence_ok'].mean()),
    })

    t = trades_df[trades_df['symbol'] == symbol] if 'trades_df' in globals() and not trades_df.empty else pd.DataFrame()
    if t.empty:
        trade_rows.append({'symbol': symbol, 'trades': 0, 'wins': 0, 'losses': 0, 'win_rate_pct': 0.0, 'tp_hits': 0, 'sl_hits': 0, 'liq_hits': 0, 'avg_rr_entry': np.nan, 'avg_loss': np.nan, 'avg_win': np.nan})
    else:
        wins = int((t['pnl'] > 0).sum())
        losses = int((t['pnl'] < 0).sum())
        trade_rows.append({
            'symbol': symbol,
            'trades': len(t),
            'wins': wins,
            'losses': losses,
            'win_rate_pct': (wins / len(t)) * 100,
            'tp_hits': int((t['exit_reason'] == 'TP').sum()),
            'sl_hits': int((t['exit_reason'] == 'SL').sum()),
            'liq_hits': int((t['exit_reason'] == 'LIQ').sum()),
            'avg_rr_entry': float(t['rr_at_entry'].mean()),
            'avg_loss': float(t.loc[t['pnl'] < 0, 'pnl'].mean()) if (t['pnl'] < 0).any() else np.nan,
            'avg_win': float(t.loc[t['pnl'] > 0, 'pnl'].mean()) if (t['pnl'] > 0).any() else np.nan,
        })

funnel_df = pd.DataFrame(funnel_rows).fillna(0)
volume_df = pd.DataFrame(volume_rows)
trade_diag_df = pd.DataFrame(trade_rows)


def _safe_rate(num, den):
    den = float(den)
    return (float(num) / den) if den > 0 else np.nan

# Setup-level rates (event-level, not per-minute window-level)
funnel_df['confirm_rate_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('confirmed', 0), r.get('sweeps', 0)) * 100, axis=1)
funnel_df['ote_pass_rate_of_confirmed_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('ote_gate_pass', 0), r.get('confirmed', 0)) * 100, axis=1)
funnel_df['ote_pass_rate_of_sweeps_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('ote_gate_pass', 0), r.get('sweeps', 0)) * 100, axis=1)
funnel_df['entry_rate_of_ote_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('setup_entered', 0), r.get('ote_gate_pass', 0)) * 100, axis=1)
funnel_df['entry_rate_of_sweeps_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('setup_entered', 0), r.get('sweeps', 0)) * 100, axis=1)
funnel_df['timeout_pre_confirm_rate_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('timeout_pre_confirm', 0), r.get('sweeps', 0)) * 100, axis=1)
funnel_df['timeout_pre_ote_rate_of_confirmed_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('timeout_pre_ote', 0), r.get('confirmed', 0)) * 100, axis=1)
funnel_df['timeout_post_ote_rate_of_ote_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('timeout_post_ote', 0), r.get('ote_gate_pass', 0)) * 100, axis=1)
funnel_df['active_unresolved_rate_of_sweeps_pct'] = funnel_df.apply(lambda r: _safe_rate(r.get('active_unresolved_end', 0), r.get('sweeps', 0)) * 100, axis=1)

# Optional: keep this to quantify window inflation pressure after OTE pass
funnel_df['entry_windows_per_ote_setup'] = funnel_df.apply(lambda r: _safe_rate(r.get('entry_windows', 0), r.get('ote_gate_pass', 0)), axis=1)

combined_diag_df = (
    trade_diag_df
    .merge(funnel_df, on='symbol', how='left')
    .merge(volume_df, on='symbol', how='left')
    .sort_values('win_rate_pct', ascending=False)
)

print("\n" + "=" * 100)
print("SYMBOL DIAGNOSTIC OVERVIEW (WIN RATE / FUNNEL / VOLUME)")
print("=" * 100)
print(combined_diag_df.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

print("\nSetup-level funnel rates:")
setup_rate_cols = [
    'symbol',
    'sweeps', 'confirmed', 'ote_gate_pass', 'setup_entered',
    'confirm_rate_pct',
    'ote_pass_rate_of_confirmed_pct',
    'ote_pass_rate_of_sweeps_pct',
    'entry_rate_of_ote_pct',
    'entry_rate_of_sweeps_pct',
    'timeout_pre_confirm_rate_pct',
    'timeout_pre_ote_rate_of_confirmed_pct',
    'timeout_post_ote_rate_of_ote_pct',
    'active_unresolved_rate_of_sweeps_pct',
    'entry_windows_per_ote_setup',
]
print(
    funnel_df[setup_rate_cols]
    .sort_values('entry_rate_of_sweeps_pct', ascending=False)
    .to_string(index=False, float_format=lambda x: f"{x:,.2f}")
)

print("\nFunnel-only view:")
funnel_cols = [
    'symbol', 'sweeps_total_all', 'sweeps_eligible_pre_dedupe', 'sweeps_removed_dedupe',
    'sweeps_eligible', 'sweeps', 'confirmed', 'ote_gate_pass',
    'entry_windows', 'entry_signals', 'setup_entered', 'sweep_score_mean', 'sweep_score_median',
    'score_ge_80', 'score_65_79', 'score_lt_65', 'rej_in_zone', 'rej_confluence', 'rej_rejection',
    'rej_tp_or_structure', 'rej_rr', 'timeout_pre_confirm', 'timeout_pre_ote',
    'timeout_post_ote', 'active_unresolved_end'
]
existing_funnel_cols = [c for c in funnel_cols if c in funnel_df.columns]
print(funnel_df[existing_funnel_cols].sort_values('setup_entered', ascending=False).to_string(index=False))

print("\nVolume-quality view:")
vol_cols = [
    'symbol', 'median_quote_volume_1m', 'mean_quote_volume_1m', 'high_participation_rate',
    'bullish_shift_ok_rate', 'bearish_shift_ok_rate', 'bullish_confluence_ok_rate', 'bearish_confluence_ok_rate'
]
print(volume_df[vol_cols].sort_values('median_quote_volume_1m', ascending=False).to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

In [ ]:
if 'trades_df' in globals() and not trades_df.empty:
    t = trades_df.copy()
    t['sl_distance_pct'] = np.where(
        t['type'] == 'SHORT',
        ((t['sl'] - t['entry']) / t['entry']) * 100,
        ((t['entry'] - t['sl']) / t['entry']) * 100,
    )

    side_exit = (
        t.groupby(['symbol', 'type', 'exit_reason'])
        .size()
        .reset_index(name='count')
        .sort_values(['symbol', 'type', 'count'], ascending=[True, True, False])
    )

    shape_diag = (
        t.groupby('symbol')
        .agg(
            trades=('pnl', 'size'),
            avg_hold_min=('hold_minutes', 'mean'),
            median_hold_min=('hold_minutes', 'median'),
            avg_sl_distance_pct=('sl_distance_pct', 'mean'),
            median_sl_distance_pct=('sl_distance_pct', 'median'),
            avg_rr_entry=('rr_at_entry', 'mean'),
        )
        .reset_index()
        .sort_values('trades', ascending=False)
    )

    print("\nTrade shape diagnostics by symbol:")
    print(shape_diag.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

    print("\nSide x exit reason counts:")
    print(side_exit.to_string(index=False))
else:
    print("No trades available for trade-shape diagnostics.")

In [ ]:
# Trade-by-trade audit log (open/close timing + rationale)
if 'trades_df' not in globals() or trades_df.empty:
    print("No closed trades to audit.")
else:
    trade_log = trades_df.copy()

    trade_log['entry_time'] = pd.to_datetime(trade_log['entry_time'], utc=True)
    trade_log['exit_time'] = pd.to_datetime(trade_log['exit_time'], utc=True)
    trade_log['hold_minutes'] = ((trade_log['exit_time'] - trade_log['entry_time']).dt.total_seconds() / 60).astype(int)

    trade_log['open_reason'] = trade_log.apply(
            lambda r: f"{r['type']} after sweep of {r['source_level_type']} + shift + OTE + confluence",
            axis=1,
    )

    exit_reason_map = {
            'TP': 'Target reached',
            'SL': 'Stop loss hit',
            'LIQ': 'Liquidation price hit',
    }
    trade_log['close_reason'] = trade_log['exit_reason'].map(exit_reason_map).fillna(trade_log['exit_reason'])

    trade_log = trade_log.sort_values('entry_time').reset_index(drop=True)
    trade_log['trade_id'] = trade_log.index + 1

    cols = [
            'trade_id', 'symbol', 'entry_time', 'type', 'entry', 'sl', 'tp',
            'open_reason', 'exit_time', 'exit_reason', 'close_reason',
            'hold_minutes', 'pnl', 'balance'
    ]

    print("\n" + "=" * 120)
    print("TRADE AUDIT LOG")
    print("=" * 120)
    print(trade_log[cols])

In [ ]:
# Analyze losing trades
if trades:
    trades_df['actual_rr'] = np.where(
        trades_df['type'] == 'SHORT',
        (trades_df['entry'] - trades_df['tp']) / (trades_df['sl'] - trades_df['entry']),
        (trades_df['tp'] - trades_df['entry']) / (trades_df['entry'] - trades_df['sl'])
    )

    losing_trades = trades_df[trades_df['pnl'] < 0]
    winning_trades = trades_df[trades_df['pnl'] > 0]
    
    print("\n🔴 LOSING TRADE ANALYSIS:")
    print(f"Total losing trades: {len(losing_trades)}")
    print(f"All hit stop loss: {(losing_trades['exit_reason'] == 'SL').sum()}")
    
    print(f"\n📊 RISK-REWARD ANALYSIS:")
    print(f"Average RR on winning trades: {winning_trades['actual_rr'].mean():.2f}:1")
    print(f"Median RR: {trades_df['actual_rr'].median():.2f}:1")
    print(f"Min RR: {trades_df['actual_rr'].min():.2f}:1")
    print(f"Max RR: {trades_df['actual_rr'].max():.2f}:1")
    
    # Check stop loss distance
    trades_df['sl_distance_pct'] = np.where(
        trades_df['type'] == 'SHORT',
        ((trades_df['sl'] - trades_df['entry']) / trades_df['entry']) * 100,
        ((trades_df['entry'] - trades_df['sl']) / trades_df['entry']) * 100
    )
    
    print(f"\n⚠️ STOP LOSS ANALYSIS:")
    print(f"Average SL distance: {trades_df['sl_distance_pct'].mean():.2f}%")
    print(f"Median SL distance: {trades_df['sl_distance_pct'].median():.2f}%")
    
    # Breakeven analysis
    win_rate = len(winning_trades) / len(trades_df)
    avg_rr = trades_df['actual_rr'].mean()
    min_win_rate_needed = 1 / (1 + avg_rr)
    
    print(f"\n💡 BREAKEVEN ANALYSIS:")
    print(f"Current win rate: {win_rate*100:.1f}%")
    print(f"Min win rate needed for breakeven: {min_win_rate_needed*100:.1f}%")
    print(f"{'✅ PROFITABLE' if win_rate > min_win_rate_needed else '❌ UNPROFITABLE'}")

## Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Plot selected primary symbol only (multi-symbol backtest is summarized in results table)
plot_symbol = 'SOL'
plot_ctx = symbol_contexts.get(plot_symbol)

if plot_ctx is None:
    raise ValueError(f"No context available for plot symbol {plot_symbol}")

plot_df = plot_ctx['df_ltf']
plot_levels = plot_ctx['levels_df']
plot_sweeps = plot_ctx['sweeps_df']

# Trade highlight toggles
HIGHLIGHT_TRADE_WINDOWS = True
ENTRY_MARKER_SIZE = 110
EXIT_MARKER_SIZE = 110

fig, ax = plt.subplots(figsize=(16, 8))

# Price
ax.plot(plot_df['timestamp'], plot_df['close'], linewidth=1, color='black', alpha=0.75, label=f'{plot_symbol} 1m close')

# Key levels
if not plot_levels.empty:
    plot_end = plot_df['timestamp'].max()
    for _, lvl in plot_levels.iterrows():
        y = lvl['price']
        x0 = lvl['created_at']
        x1 = lvl['swept_at'] if pd.notna(lvl['swept_at']) else plot_end

        if lvl['level_type'].startswith('prev_session'):
            color, lw, alpha = ('#6a1b9a', 1.4, 0.6)
        elif lvl['level_type'].startswith('prev_h4'):
            color, lw, alpha = ('#1565c0', 1.1, 0.5)
        else:
            color, lw, alpha = ('#2e7d32', 0.9, 0.35)

        ls = '-' if lvl['side'] == 'high' else '--'
        ax.hlines(y=y, xmin=x0, xmax=x1, colors=color, linestyles=ls, linewidth=lw, alpha=alpha)

# Sweep markers (eligible/reliable sweeps only)
if not plot_sweeps.empty:
    short_sw = plot_sweeps[plot_sweeps['direction'] == 'SHORT']
    long_sw = plot_sweeps[plot_sweeps['direction'] == 'LONG']

    has_score = 'reliability_score' in plot_sweeps.columns

    if not short_sw.empty:
        if has_score:
            short_sizes = 30 + short_sw['reliability_score'].clip(0, 100) * 0.8
            ax.scatter(short_sw['timestamp'], short_sw['level_price'], marker='v', color='red', s=short_sizes, alpha=0.65, label='High sweep (reliable)')
        else:
            ax.scatter(short_sw['timestamp'], short_sw['level_price'], marker='v', color='red', s=40, alpha=0.8, label='High sweep')

    if not long_sw.empty:
        if has_score:
            long_sizes = 30 + long_sw['reliability_score'].clip(0, 100) * 0.8
            ax.scatter(long_sw['timestamp'], long_sw['level_price'], marker='^', color='green', s=long_sizes, alpha=0.65, label='Low sweep (reliable)')
        else:
            ax.scatter(long_sw['timestamp'], long_sw['level_price'], marker='^', color='green', s=40, alpha=0.8, label='Low sweep')

# Trade markers for plotted symbol only
if 'trades_df' in globals() and not trades_df.empty:
    symbol_trades = trades_df[trades_df['symbol'] == plot_symbol].sort_values('entry_time')

    # Add one-time legend labels for entry/exit markers
    added_entry_label = False
    added_exit_label = False

    for _, tr in symbol_trades.iterrows():
        color = 'green' if tr['pnl'] > 0 else 'red'
        marker = '^' if tr['type'] == 'LONG' else 'v'

        if tr['exit_reason'] == 'TP':
            exit_price = tr['tp']
        elif tr['exit_reason'] == 'SL':
            exit_price = tr['sl']
        else:
            exit_price = tr['liq_price']

        # Optional window highlight between entry and exit times
        if HIGHLIGHT_TRADE_WINDOWS:
            ax.axvspan(tr['entry_time'], tr['exit_time'], color=color, alpha=0.05, linewidth=0)

        # Entry/exit points
        ax.scatter(
            tr['entry_time'],
            tr['entry'],
            color=color,
            marker=marker,
            s=ENTRY_MARKER_SIZE,
            zorder=6,
            label='Trade entry' if not added_entry_label else None,
        )
        added_entry_label = True

        ax.scatter(
            tr['exit_time'],
            exit_price,
            color=color,
            marker='x',
            s=EXIT_MARKER_SIZE,
            zorder=6,
            label='Trade exit' if not added_exit_label else None,
        )
        added_exit_label = True

        # Connect entry -> exit
        ax.plot(
            [tr['entry_time'], tr['exit_time']],
            [tr['entry'], exit_price],
            color=color,
            linestyle=':',
            linewidth=1,
            alpha=0.7,
        )

ax.set_title(f'ICT Key Levels, Reliable Sweeps, and Trades ({plot_symbol} 1m)', fontsize=14, fontweight='bold')
ax.set_xlabel('UTC Time')
ax.set_ylabel('Price')
ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
ax.legend(loc='best')
plt.tight_layout()
plt.show()